In [1]:
import pickle
import re
import unicodedata
import tqdm
from datasets import load_dataset
import os

In [2]:
sampa_to_ipa = {
    "2": "ø",
    "9": "œ",
    "@": "ə",
    "E": "ɛ",
    "E/": ("e", "ɛ"),        # close or open-mid front
    "H": "ɥ",
    "J": "ɲ",
    "O": "ɔ",
    "O/": ("o", "ɔ"),        # close or open-mid back
    "R": "ʁ",
    "S": "ʃ",
    "U~/": ("ɛ̃", "œ̃"),     # indeterminate nasal front-mid
    "Z": "ʒ",
    "a": "a",
    "a~": "ɑ̃",
    "b": "b",
    "d": "d",
    "e": "e",
    "f": "f",
    "g": "g",
    "i": "i",
    "j": "j",
    "k": "k",
    "l": "l",
    "m": "m",
    "n": "n",
    "o": "o",
    "o~": "ɔ̃",
    "p": "p",
    "s": "s",
    "t": "t",
    "u": "u",
    "v": "v",
    "w": "w",
    "y": "y",
    "z": "z",
}

In [3]:
def text2words(text, remove_accents=False):
    # 1. Normalize whitespace
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text)

    # 2. Lowercase
    text = text.lower()

    # 3. Optionally strip accents (é → e, etc.)
    if remove_accents:
        text = ''.join(
            c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) != 'Mn'
        )

    # 4. Remove punctuation/numbers (keep letters and apostrophes)
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ'\- ]", " ", text)

    # 5. Tokenize into words
    words = text.split()

    return words

In [4]:
reload_ds = True
if os.path.exists("word_counts_fr.pkl") and not reload_ds:
    with open("word_counts_fr.pkl", "rb") as f:
        word_counts = pickle.load(f)
else:
    ds = load_dataset("PleIAs/French-PD-Books", split="train")
    print(ds)
    print(ds.column_names)

    word_counts = {}
    for book in tqdm.tqdm(ds.select(range(0, 10000))):
        words = text2words(book["complete_text"])
        for word in words:
            word_counts[word] = word_counts.get(word, 0) + 1
    with open("word_counts_fr.pkl", "wb") as f:
        pickle.dump(word_counts, f)

Resolving data files:   0%|          | 0/145 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/145 [00:00<?, ?it/s]

Dataset({
    features: ['file_id', 'ocr', 'title', 'date', 'author', 'page_count', 'word_count', 'character_count', 'complete_text'],
    num_rows: 289577
})
['file_id', 'ocr', 'title', 'date', 'author', 'page_count', 'word_count', 'character_count', 'complete_text']


100%|██████████| 10000/10000 [03:57<00:00, 42.11it/s]


In [5]:
import sys
sys.path.insert(0, "../..")

from utils import ipa_processing

In [6]:
from phonemizer import phonemize
words = [word for word, count in word_counts.items()]
phonemized_words = phonemize(words, language="fr-fr", backend="espeak", strip=True, njobs=8)
pronunciation_dict = {word: ipa for word, ipa in zip(words, phonemized_words)}

In [8]:
with open("pronunciation_dict_fr.pkl", "wb") as f:
    pickle.dump(pronunciation_dict, f)

In [7]:
pronunciation_counts = {}

for word, count in tqdm.tqdm(word_counts.items()):
    pronunciation = ipa_processing.word2ipa(word, lang="fr", pronunciation_dict=pronunciation_dict)
    if pronunciation:
        pronunciation_counts[pronunciation] = pronunciation_counts.get(pronunciation, 0) + count

with open("pronunciation_counts_fr.pkl", "wb") as f:
    pickle.dump(pronunciation_counts, f)

100%|██████████| 9845680/9845680 [00:20<00:00, 474281.00it/s]
